# Prepare and segment reviews

This notebook is part of the public reproducibility workflow for the study.

Public workflow stage: 01  
Purpose: Normalize review text, join paper metadata, segment sentences, and remove noise.

Source notebooks:
- `data_preprocess.ipynb` (formal NFKC routine)
- `分句.ipynb` (final join, segmentation, and noise-removal branches)

Inputs: `data/sample/sample_reviews.jsonl`, `data/sample/sample_papers.csv` (provided in a later release batch).  
Outputs: `data/work/review_level.parquet`, `data/work/sentence_parts/`.


In [ ]:
from pathlib import Path
import sys

# Jupyter may start in the repository root or its notebooks directory.
_candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
_root = next((p for p in _candidates if (p / "src" / "project_paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory")
sys.path.insert(0, str(_root))
from src.project_paths import SAMPLE_DATA_DIR, WORK_DATA_DIR, CONFIG_DIR, project_path

try:
    import yaml
except ImportError as exc:
    raise ImportError("Install PyYAML to read config/config.example.yaml") from exc
with (CONFIG_DIR / "config.example.yaml").open(encoding="utf-8") as _f:
    config = yaml.safe_load(_f)
sample_dir = project_path(config["paths"]["sample_data"])
work_dir = project_path(config["paths"]["work_data"])

import json
import re
import unicodedata
import pandas as pd
from tqdm.auto import tqdm

review_path = sample_dir / "sample_reviews.jsonl"
paper_path = sample_dir / "sample_papers.csv"
for path in (review_path, paper_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required sample input is not available yet: {path}")
sentence_dir = work_dir / "sentence_parts"
sentence_dir.mkdir(parents=True, exist_ok=True)
DO_NFKC = bool(config["review_processing"]["text_normalization"])
MIN_LEN = int(config["review_processing"]["min_sentence_length"])
CHUNK_SIZE = int(config["review_processing"]["chunk_size"])


## Normalize and adapt source columns

The adapter accepts both the formal source schema and the public sample schema. Only the source-schema keys below contain non-English text.


In [ ]:
_CONTROL_CHARS = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")

def normalize_nfkc(value):
    if isinstance(value, str):
        value = unicodedata.normalize("NFKC", value.replace("\u00A0", " "))
        return _CONTROL_CHARS.sub("", value)
    return value

# ALLOWED_SOURCE_SCHEMA_LITERAL: formal source-column names only.
COLUMN_MAP = {
    "论文编码": "paper_code", "学科领域": "subject_code",
    "发表时间": "pub_date", "标题": "title", "主题": "topic",
    "ID": "review_id", "Reviewer": "reviewer_tag",
    "审稿意见文本": "content_raw",
}

def adapt_columns(frame):
    frame = frame.copy()
    frame.columns = [normalize_nfkc(col) for col in frame.columns]
    for col in frame.select_dtypes(include=["object"]).columns:
        frame[col] = frame[col].map(normalize_nfkc)
    return frame.rename(columns=COLUMN_MAP)

papers = adapt_columns(pd.read_csv(paper_path))
reviews = adapt_columns(pd.read_json(review_path, lines=True))
if "content_raw" not in reviews and "content" in reviews:
    reviews = reviews.rename(columns={"content": "content_raw"})
required_paper = {"paper_code", "subject_code", "pub_date", "title", "topic"}
required_review = {"review_id", "paper_code", "reviewer_tag", "content_raw"}
if missing := required_paper.difference(papers.columns):
    raise KeyError(f"Paper input is missing columns: {sorted(missing)}")
if missing := required_review.difference(reviews.columns):
    raise KeyError(f"Review input is missing columns: {sorted(missing)}")
papers["pub_date"] = pd.to_datetime(papers["pub_date"], errors="coerce")
papers["year"] = papers["pub_date"].dt.year
# The final source join retains the latest publication-date row per paper.
papers = papers.sort_values("pub_date").drop_duplicates("paper_code", keep="last")
reviews = reviews.merge(
    papers[["paper_code", "subject_code", "pub_date", "year", "title", "topic"]],
    on="paper_code", how="left", validate="many_to_one"
)
reviews["subject_code"] = reviews["subject_code"].astype("string")
print(f"Joined {len(reviews)} reviews to {papers['paper_code'].nunique()} papers")


## Segment text

Punkt is preferred when its local resources are available; the source regular-expression fallback is used otherwise. No corpus download is performed automatically.


In [ ]:
def setup_sentence_tokenizer():
    fallback = re.compile(r"(?<=[.!?])\s+(?=(?:[A-Z]|\(|\[|\"|'|\d))")
    def regex_tokenize(text):
        return [item.strip() for item in fallback.split(text.strip()) if item.strip()]
    try:
        import nltk
        from nltk.tokenize import sent_tokenize
        nltk.data.find("tokenizers/punkt")
        return sent_tokenize, "nltk_punkt"
    except (ImportError, LookupError):
        return regex_tokenize, "regex_fallback"

sent_tokenize_fn, TOKENIZER_NAME = setup_sentence_tokenizer()
LIST_RE = re.compile(r"^\s*(?:[-•*]+|\(?\d{1,3}\)?[.)]|[A-Za-z][.)])\s+")
MULTI_SPACE_RE = re.compile(r"[ \t]+")
MULTI_NL_RE = re.compile(r"\n{3,}")
ABBR = ["Fig.", "Figs.", "Eq.", "Eqs.", "Ref.", "Refs.", "et al.", "i.e.", "e.g.", "vs.", "Dr.", "Prof.", "Mr.", "Ms.", "Sr.", "Jr.", "U.S.", "U.K.", "No."]
ABBR_RE = re.compile(r"\b(" + "|".join(re.escape(item) for item in ABBR) + r")")

def light_normalize(text):
    if text is None:
        return ""
    value = str(text).replace("\r\n", "\n").replace("\r", "\n")
    value = MULTI_NL_RE.sub("\n\n", value)
    value = MULTI_SPACE_RE.sub(" ", value)
    return unicodedata.normalize("NFKC", value).strip() if DO_NFKC else value.strip()

def split_to_blocks(content_norm):
    blocks = []
    for para_idx, paragraph in enumerate(re.split(r"\n\s*\n+", content_norm)):
        lines = [line.rstrip() for line in paragraph.strip().split("\n") if line.strip()]
        buf = []
        for line in lines:
            if LIST_RE.match(line):
                if buf:
                    blocks.append((" ".join(buf).strip(), para_idx, False))
                    buf = []
                blocks.append((line.strip(), para_idx, True))
            else:
                buf.append(line.strip())
        if buf:
            blocks.append((" ".join(buf).strip(), para_idx, False))
    return blocks

def sentence_split_one_review(content_raw):
    content_norm = light_normalize(content_raw)
    rows = []
    noise_count = 0
    for block_idx, (block, para_idx, is_list_item) in enumerate(split_to_blocks(content_norm)):
        if not block:
            continue
        protected = ABBR_RE.sub(lambda match: match.group(1).replace(".", "<DOT>"), block)
        try:
            sentences = sent_tokenize_fn(protected)
        except Exception:
            sentences = re.split(r"(?<=[.!?])\s+", protected)
        for sentence in sentences:
            sentence = sentence.replace("<DOT>", ".").strip()
            if not sentence:
                continue
            reason = ""
            if len(sentence) < MIN_LEN:
                reason = "too_short"
            elif re.fullmatch(r"[\W_]+", sentence):
                reason = "only_symbols"
            elif not re.search(r"[A-Za-z]", sentence):
                reason = "no_alpha"
            noise_count += bool(reason)
            rows.append(dict(sent_idx=len(rows), para_idx=para_idx, block_idx=block_idx,
                             is_list_item=is_list_item, sentence_clean=sentence,
                             is_noise=bool(reason), noise_reason=reason))
    stats = dict(n_sentences_total=len(rows), n_sentences_noise=noise_count,
                 n_sentences_kept=len(rows) - noise_count)
    return content_norm, rows, stats


## Write the final, noise-filtered outputs

The final source branch removes noise and reindexes sentence IDs. This public workflow writes the final form directly into its new work directory; it does not back up or replace any original research files.


In [ ]:
review_rows = []
sentence_columns = ["review_id", "paper_code", "reviewer_tag", "subject_code", "pub_date", "year",
                    "sent_idx", "para_idx", "block_idx", "is_list_item", "sentence_clean"]
for part, start in enumerate(tqdm(range(0, len(reviews), CHUNK_SIZE), desc="Segment reviews")):
    sentence_rows = []
    for row in reviews.iloc[start:start + CHUNK_SIZE].itertuples(index=False):
        content_norm, sentences, stats = sentence_split_one_review(row.content_raw)
        review_rows.append(dict(paper_code=row.paper_code, review_id=row.review_id,
                                reviewer_tag=row.reviewer_tag, subject_code=row.subject_code,
                                pub_date=row.pub_date, year=row.year, title=row.title,
                                topic=row.topic, content_raw=row.content_raw,
                                content_norm=content_norm, n_sentences_total=stats["n_sentences_kept"],
                                n_sentences_noise=0, n_sentences_kept=stats["n_sentences_kept"],
                                n_sentences_deleted_noise=stats["n_sentences_noise"]))
        kept = [sentence for sentence in sentences if not sentence["is_noise"]]
        for new_idx, sentence in enumerate(kept):
            sentence_rows.append(dict(review_id=row.review_id, paper_code=row.paper_code,
                                      reviewer_tag=row.reviewer_tag, subject_code=row.subject_code,
                                      pub_date=row.pub_date, year=row.year, sent_idx=new_idx,
                                      para_idx=sentence["para_idx"], block_idx=sentence["block_idx"],
                                      is_list_item=sentence["is_list_item"],
                                      sentence_clean=sentence["sentence_clean"]))
    pd.DataFrame(sentence_rows, columns=sentence_columns).to_parquet(
        sentence_dir / f"part-{part:05d}.parquet", index=False)
pd.DataFrame(review_rows).to_parquet(work_dir / "review_level.parquet", index=False)
(work_dir / "segmentation_metadata.json").write_text(
    json.dumps(dict(tokenizer=TOKENIZER_NAME, do_nfkc=DO_NFKC,
                    min_len=MIN_LEN, chunk_size=CHUNK_SIZE,
                    parts_written=(len(reviews) + CHUNK_SIZE - 1) // CHUNK_SIZE), indent=2),
    encoding="utf-8")
print(f"Saved review-level and sentence-part outputs under {work_dir}")
